# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shrishagk/My_flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [4]:
# ML-09 — Section 1: Two paper findings + methodology questions

finding_1 = """
Finding 1 — CTR/position signals can be informative, but raw CTR is not automatically an unbiased quality signal

The research review reports that SEO practitioner systems commonly use CTR and position as signals, while click-model research shows that observed clicks are affected by position/examination bias. Therefore, a methodology that treats raw CTR as a direct measure of content quality should explain how position effects are handled.

Where does the label come from?

In the FlyRank modeling work, the target used in ML-08 is not a human-annotated "needs refresh" label. It is a proxy: next-day GSC clicks. This is an observed future performance outcome, not a direct annotation of whether a page should actually be refreshed.

Does the validation design carry the claim?

Only partially. A time-aware split is appropriate because the target occurs after the features. However, a five-day January window is too short to establish that the model predicts genuine content decay or refresh need. It only tests whether current performance signals can rank observations by their subsequent next-day click outcomes in this small evaluation period.

Methodology question:

Would the ranking remain useful when evaluated over a substantially longer time period, across multiple clients, and with a target that more directly represents sustained content-performance deterioration?
"""

finding_2 = """
Finding 2 — There is limited established academic methodology for content-refresh prioritization

The research review found a gap around formally treating content-refresh prioritization as a machine-learning decision problem under a finite editorial budget. Existing practitioner approaches are commonly heuristic or opaque, while causal evaluation of actual refresh interventions remains insufficiently established.

Where does the label come from?

This is an important distinction: the research-gap claim itself is not a supervised ML label. It comes from the literature review. In the ML-08 experiment, the practical ranking target is instead next-day clicks because the available warehouse data does not contain a validated human label indicating that a page genuinely required a refresh.

Does the validation design carry the claim?

No, not by itself. A Random Forest beating a rule baseline on NDCG@10 would show that the model ranked the chosen proxy outcome better in this evaluation window. It would not establish that the model identifies pages that genuinely need refreshing, nor that refreshing those pages would improve performance.

Methodology question:

A stronger study needs a validated refresh/outcome definition and a longer temporal evaluation. Ideally, actual refresh events and their timestamps would allow the model to be evaluated against meaningful future outcomes rather than only next-day clicks.
"""

methodology_conclusion = """
Methodology conclusion

The two findings point to the same methodological issue: the target must be separated from the business decision.

For this notebook, I therefore treat next-day clicks as a proxy outcome, not as a refresh label. The model can be evaluated as a ranking system, but the result should remain directional decision-support evidence, not evidence that the model has learned causal content decay or the correct refresh action.
"""

print(finding_1)
print("=" * 80)
print(finding_2)
print("=" * 80)
print(methodology_conclusion)


Finding 1 — CTR/position signals can be informative, but raw CTR is not automatically an unbiased quality signal

The research review reports that SEO practitioner systems commonly use CTR and position as signals, while click-model research shows that observed clicks are affected by position/examination bias. Therefore, a methodology that treats raw CTR as a direct measure of content quality should explain how position effects are handled.

Where does the label come from?

In the FlyRank modeling work, the target used in ML-08 is not a human-annotated "needs refresh" label. It is a proxy: next-day GSC clicks. This is an observed future performance outcome, not a direct annotation of whether a page should actually be refreshed.

Does the validation design carry the claim?

Only partially. A time-aware split is appropriate because the target occurs after the features. However, a five-day January window is too short to establish that the model predicts genuine content decay or refresh ne

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
from getpass import getpass

import duckdb
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import ndcg_score
from sklearn.model_selection import train_test_split

HF_TOKEN = getpass("Enter your Hugging Face token: ")

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

REL = "hf://datasets/FlyRank/internship-warehouse"

PERF_JAN = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2025-01/*.parquet'"
    f")"
)

data = con.sql(
    f"SELECT * FROM {PERF_JAN}"
).df()

data["report_date"] = pd.to_datetime(
    data["report_date"]
)

print("Shape:", data.shape)
print("Date range:",
      data["report_date"].min().date(),
      "to",
      data["report_date"].max().date())

print("\nRows by date:")
print(
    data["report_date"]
    .value_counts()
    .sort_index()
)

Shape: (1297, 31)
Date range: 2025-01-27 to 2025-01-31

Rows by date:
report_date
2025-01-27    303
2025-01-28    317
2025-01-29    262
2025-01-30    194
2025-01-31    221
Name: count, dtype: int64


In [6]:
group_cols = [
    "client_hash_id",
    "content_hash_id"
]

duplicate_grain = (
    data.groupby(
        group_cols + ["report_date"]
    )
    .size()
)

print(
    "Duplicate client-content-date rows:",
    (duplicate_grain > 1).sum()
)

assert (duplicate_grain > 1).sum() == 0

Duplicate client-content-date rows: 0


In [7]:
data = data.sort_values(
    group_cols + ["report_date"]
).copy()

data["next_date"] = (
    data.groupby(group_cols)["report_date"]
    .shift(-1)
)

data["days_to_next"] = (
    data["next_date"] -
    data["report_date"]
).dt.days

data["next_day_clicks"] = (
    data.groupby(group_cols)["gsc_clicks"]
    .shift(-1)
)

data.loc[
    data["days_to_next"] != 1,
    "next_day_clicks"
] = np.nan

print("Days to next observation:")
print(
    data["days_to_next"]
    .value_counts(dropna=False)
    .sort_index()
)

model_data = data.dropna(
    subset=["next_day_clicks"]
).copy()

print(
    "\nValid next-day target rows:",
    len(model_data)
)

Days to next observation:
days_to_next
1.0    780
2.0     34
3.0      5
4.0      2
NaN    476
Name: count, dtype: int64

Valid next-day target rows: 780


In [8]:
model_data["ctr"] = np.where(
    model_data["gsc_impressions"] > 0,
    model_data["gsc_clicks"]
    / model_data["gsc_impressions"],
    np.nan
)

model_data["total_sessions"] = (
    model_data["sessions_organic"].fillna(0)
    + model_data["sessions_direct"].fillna(0)
    + model_data["sessions_referral"].fillna(0)
    + model_data["sessions_social"].fillna(0)
    + model_data["sessions_paid"].fillna(0)
    + model_data["sessions_ai"].fillna(0)
)

model_data["organic_share"] = np.where(
    model_data["total_sessions"] > 0,
    model_data["sessions_organic"]
    / model_data["total_sessions"],
    np.nan
)

features = [
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events",
]

X = model_data[features].copy()
y = model_data["next_day_clicks"].copy()

# Random Forest does not accept NaN.
X = X.fillna(0)

print("Number of features:", len(features))
print("X shape:", X.shape)
print("y shape:", y.shape)

Number of features: 14
X shape: (780, 14)
y shape: (780,)


In [9]:
def make_model():
    return RandomForestRegressor(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

print(make_model())

RandomForestRegressor(max_depth=6, min_samples_leaf=5, n_estimators=300,
                      n_jobs=-1, random_state=42)


In [10]:
random_train_idx, random_test_idx = train_test_split(
    np.arange(len(model_data)),
    test_size=0.20,
    random_state=42
)

X_random_train = X.iloc[random_train_idx]
X_random_test = X.iloc[random_test_idx]

y_random_train = y.iloc[random_train_idx]
y_random_test = y.iloc[random_test_idx]

random_model = make_model()

random_model.fit(
    X_random_train,
    y_random_train
)

random_pred = random_model.predict(
    X_random_test
)

random_ndcg = ndcg_score(
    [y_random_test.to_numpy()],
    [random_pred],
    k=min(10, len(y_random_test))
)

print("Random-split rows:")
print("Train:", len(X_random_train))
print("Test :", len(X_random_test))

print(
    "\nRandom-split NDCG@10:",
    round(random_ndcg, 6)
)

Random-split rows:
Train: 624
Test : 156

Random-split NDCG@10: 0.207261


In [11]:
train_end = pd.Timestamp(
    "2025-01-29"
)

time_train_mask = (
    model_data["report_date"]
    <= train_end
)

time_test_mask = (
    model_data["report_date"]
    > train_end
)

X_time_train = X.loc[time_train_mask]
X_time_test = X.loc[time_test_mask]

y_time_train = y.loc[time_train_mask]
y_time_test = y.loc[time_test_mask]

time_model = make_model()

time_model.fit(
    X_time_train,
    y_time_train
)

time_pred = time_model.predict(
    X_time_test
)

time_ndcg = ndcg_score(
    [y_time_test.to_numpy()],
    [time_pred],
    k=min(10, len(y_time_test))
)

print("Time-aware rows:")
print("Train:", len(X_time_train))
print("Test :", len(X_time_test))

print(
    "\nTime-aware NDCG@10:",
    round(time_ndcg, 6)
)

print(
    "\nTraining period:",
    model_data.loc[
        time_train_mask,
        "report_date"
    ].min().date(),
    "to",
    model_data.loc[
        time_train_mask,
        "report_date"
    ].max().date()
)

print(
    "Evaluation period:",
    model_data.loc[
        time_test_mask,
        "report_date"
    ].min().date(),
    "to",
    model_data.loc[
        time_test_mask,
        "report_date"
    ].max().date()
)

Time-aware rows:
Train: 600
Test : 180

Time-aware NDCG@10: 0.689987

Training period: 2025-01-27 to 2025-01-29
Evaluation period: 2025-01-30 to 2025-01-30


In [12]:
validation_comparison = pd.DataFrame({
    "split": [
        "Random split",
        "Time-aware split"
    ],
    "NDCG@10": [
        random_ndcg,
        time_ndcg
    ]
})

display(validation_comparison)

,split,NDCG@10
0,Random split,0.207261
1,Time-aware split,0.689987


In [13]:
gap = random_ndcg - time_ndcg

print(
    "Random - time-aware NDCG@10 gap:",
    round(gap, 6)
)

if gap > 0:
    print(
        "\nThe random split scores higher than the "
        "time-aware split. This is consistent with "
        "the possibility that random splitting benefits "
        "from temporal overlap between training and "
        "evaluation observations."
    )
else:
    print(
        "\nThe random split did not score higher than "
        "the time-aware split in this run."
    )

Random - time-aware NDCG@10 gap: -0.482726

The random split did not score higher than the time-aware split in this run.


In [14]:
train_data = model_data.loc[
    time_train_mask
].copy()

eval_data = model_data.loc[
    time_test_mask
].copy()

ctr_threshold = train_data[
    "ctr"
].quantile(0.25)

position_threshold = train_data[
    "gsc_avg_position"
].quantile(0.75)

print("Training-derived thresholds:")
print(
    "CTR:",
    ctr_threshold
)

print(
    "Position:",
    position_threshold
)

Training-derived thresholds:
CTR: 0.0
Position: 48.962500000000006


In [15]:
history = data.sort_values(
    group_cols + ["report_date"]
).copy()

history["previous_day_clicks"] = (
    history.groupby(group_cols)["gsc_clicks"]
    .shift(1)
)

previous_clicks = history[
    group_cols
    + ["report_date", "previous_day_clicks"]
].copy()

eval_data = eval_data.merge(
    previous_clicks,
    on=group_cols + ["report_date"],
    how="left",
    sort=False
)

In [16]:
eval_data["baseline_score"] = 0

# Declining current-day clicks versus previous day.
trend_down = (
    eval_data["previous_day_clicks"].notna()
    & (
        eval_data["gsc_clicks"]
        < eval_data["previous_day_clicks"]
    )
)

eval_data.loc[
    trend_down,
    "baseline_score"
] += 3

# Low CTR.
eval_data.loc[
    eval_data["ctr"] <= ctr_threshold,
    "baseline_score"
] += 2

# Weak observed position.
eval_data.loc[
    eval_data["gsc_avg_position"]
    >= position_threshold,
    "baseline_score"
] += 2

print(
    eval_data["baseline_score"]
    .value_counts()
    .sort_index()
)

baseline_score
0     16
2    105
4     47
5     12
Name: count, dtype: int64


In [17]:
eval_data["model_score"] = time_pred

assert len(eval_data) == len(time_pred)
assert eval_data["model_score"].notna().all()

In [18]:
y_true = eval_data[
    "next_day_clicks"
].to_numpy()

baseline_scores = eval_data[
    "baseline_score"
].to_numpy()

model_scores = eval_data[
    "model_score"
].to_numpy()

k = min(
    10,
    len(eval_data)
)

baseline_ndcg = ndcg_score(
    [y_true],
    [baseline_scores],
    k=k
)

model_ndcg = ndcg_score(
    [y_true],
    [model_scores],
    k=k
)

comparison = pd.DataFrame({
    "method": [
        "Rule baseline",
        "Random Forest"
    ],
    "NDCG@10": [
        baseline_ndcg,
        model_ndcg
    ]
})

display(comparison)

,method,NDCG@10
0,Rule baseline,0.066051
1,Random Forest,0.689987


In [19]:
zero_click_rate = (
    y_time_test.eq(0).mean()
)

print(
    "Evaluation observations:",
    len(y_time_test)
)

print(
    "Zero next-day-click observations:",
    int(y_time_test.eq(0).sum())
)

print(
    "Zero next-day-click proportion:",
    round(zero_click_rate, 4)
)

print(
    "\nNext-day click summary:"
)

print(
    y_time_test.describe()
)

Evaluation observations: 180
Zero next-day-click observations: 171
Zero next-day-click proportion: 0.95

Next-day click summary:
count    180.000000
mean       0.105556
std        0.554285
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        5.000000
Name: next_day_clicks, dtype: float64


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [20]:
print("Final model features:")
for feature in features:
    print("-", feature)

assert "next_day_clicks" not in features
assert "next_date" not in features
assert "days_to_next" not in features

print("\nTarget-derived timing fields are excluded.")

Final model features:
- gsc_impressions
- gsc_clicks
- ctr
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_engaged_sessions
- sessions_organic
- sessions_direct
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai
- scroll_events

Target-derived timing fields are excluded.


In [21]:
identifier_columns = [
    "client_hash_id",
    "content_hash_id",
    "report_date"
]

identifier_in_features = [
    col for col in identifier_columns
    if col in features
]

print(
    "Identifiers accidentally used as features:",
    identifier_in_features
)

assert identifier_in_features == []

Identifiers accidentally used as features: []


In [22]:
product_like_columns = [
    col for col in data.columns
    if any(
        term in col.lower()
        for term in [
            "score",
            "rank",
            "priority",
            "action",
            "refresh",
            "flag",
            "recommendation"
        ]
    )
]

print(
    "Potential product/decision columns in raw data:"
)

print(product_like_columns)

used_product_columns = [
    col for col in features
    if col in product_like_columns
]

print(
    "\nPotential product columns used by model:",
    used_product_columns
)

assert used_product_columns == []

Potential product/decision columns in raw data:
[]

Potential product columns used by model: []


In [23]:
future_fields = [
    "next_date",
    "days_to_next",
    "next_day_clicks"
]

future_fields_in_features = [
    col for col in future_fields
    if col in features
]

print(
    "Future/target fields accidentally used:",
    future_fields_in_features
)

assert future_fields_in_features == []

Future/target fields accidentally used: []


In [24]:
print(
    "GA4 availability distribution:"
)

print(
    data["ga4_data_available"]
    .value_counts(dropna=False)
)

print(
    "\nGSC availability distribution:"
)

print(
    data["gsc_data_available"]
    .value_counts(dropna=False)
)

GA4 availability distribution:
ga4_data_available
False    1297
Name: count, dtype: int64

GSC availability distribution:
gsc_data_available
True    1297
Name: count, dtype: int64


In [25]:
missingness = (
    model_data[features]
    .isna()
    .mean()
    .sort_values(
        ascending=False
    )
)

display(
    missingness.to_frame(
        "missing_fraction"
    )
)

,missing_fraction
gsc_impressions,0.0
gsc_clicks,0.0
ctr,0.0
gsc_avg_position,0.0
ga4_pageviews,0.0
ga4_sessions,0.0
ga4_engaged_sessions,0.0
sessions_organic,0.0
sessions_direct,0.0
sessions_referral,0.0


In [26]:
X_leaky = X.copy()

X_leaky["LEAKY_NEXT_DAY_CLICKS"] = y.to_numpy()

leaky_train = X_leaky.loc[
    time_train_mask
]

leaky_test = X_leaky.loc[
    time_test_mask
]

leaky_model = make_model()

leaky_model.fit(
    leaky_train,
    y_time_train
)

leaky_pred = leaky_model.predict(
    leaky_test
)

leaky_ndcg = ndcg_score(
    [y_time_test.to_numpy()],
    [leaky_pred],
    k=min(
        10,
        len(y_time_test)
    )
)

print(
    "Honest model NDCG@10:",
    round(model_ndcg, 6)
)

print(
    "Deliberately leaky model NDCG@10:",
    round(leaky_ndcg, 6)
)

Honest model NDCG@10: 0.689987
Deliberately leaky model NDCG@10: 0.987458


In [27]:
leakage_checks = {
    "Target excluded": "next_day_clicks" not in features,
    "Future date excluded": "next_date" not in features,
    "Gap field excluded": "days_to_next" not in features,
    "IDs excluded": identifier_in_features == [],
    "Product scores excluded": used_product_columns == [],
    "Leaky diagnostic run": True,
}

leakage_table = pd.DataFrame({
    "check": list(leakage_checks.keys()),
    "passed": list(leakage_checks.values())
})

display(leakage_table)

assert all(leakage_checks.values())

,check,passed
0,Target excluded,True
1,Future date excluded,True
2,Gap field excluded,True
3,IDs excluded,True
4,Product scores excluded,True
5,Leaky diagnostic run,True


In [28]:
importance = pd.DataFrame({
    "feature": features,
    "importance": time_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(importance)

,feature,importance
0,gsc_impressions,0.759139
3,gsc_avg_position,0.227536
2,ctr,0.012069
1,gsc_clicks,0.001256
4,ga4_pageviews,0.000000
5,ga4_sessions,0.000000
6,ga4_engaged_sessions,0.000000
7,sessions_organic,0.000000
8,sessions_direct,0.000000
9,sessions_referral,0.000000


In [29]:
top_features = importance.head(5)

print(
    "Top five model features:"
)

display(top_features)

Top five model features:


,feature,importance
0,gsc_impressions,0.759139
3,gsc_avg_position,0.227536
2,ctr,0.012069
1,gsc_clicks,0.001256
4,ga4_pageviews,0.000000


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [30]:
print("=" * 60)
print("ML-09 VALIDATION AND CLAIM AUDIT")
print("=" * 60)

print(
    "\nValid next-day target rows:",
    len(model_data)
)

print(
    "Random split NDCG@10:",
    round(random_ndcg, 6)
)

print(
    "Time-aware NDCG@10:",
    round(time_ndcg, 6)
)

print(
    "Rule baseline NDCG@10:",
    round(baseline_ndcg, 6)
)

print(
    "Random Forest NDCG@10:",
    round(model_ndcg, 6)
)

print(
    "Zero-target proportion:",
    round(zero_click_rate, 4)
)

print(
    "\nLeaky diagnostic NDCG@10:",
    round(leaky_ndcg, 6)
)

print(
    "\nTop model features:"
)

display(
    importance.head(5)
)

ML-09 VALIDATION AND CLAIM AUDIT

Valid next-day target rows: 780
Random split NDCG@10: 0.207261
Time-aware NDCG@10: 0.689987
Rule baseline NDCG@10: 0.066051
Random Forest NDCG@10: 0.689987
Zero-target proportion: 0.95

Leaky diagnostic NDCG@10: 0.987458

Top model features:


,feature,importance
0,gsc_impressions,0.759139
3,gsc_avg_position,0.227536
2,ctr,0.012069
1,gsc_clicks,0.001256
4,ga4_pageviews,0.000000


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.